In [2]:
import torch, math
import torch.nn as nn
import torch.nn.functional as F

Mapping network: z → w

In [3]:
class Mapping(nn.Module):
    def __init__(self, zdim=512, wdim=512, n_layers=8):
        super().__init__()
        layers = []
        for i in range(n_layers):
            layers += [nn.Linear(zdim if i == 0 else wdim, wdim), nn.LeakyReLU(0.2)]
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        z = z * torch.rsqrt(z.pow(2).mean(dim=1, keepdim=True) + 1e-8)  # normalize z
        return self.net(z)

z = torch.randn(3, 512)     # include the batch dimension
w = Mapping()(z)            # (B, 512)
w.shape

torch.Size([3, 512])

w is the latent space where you can do latent walking (cleaner)

Synthesis network (z -> image)

In [4]:
class ModConv(nn.Module):
	def __init__(self, in_ch, out_ch, wdim=512, k=3, demod=True):
		super().__init__()
		self.wdim, self.demod, self.k = wdim, demod, k
		self.in_ch, self.out_ch = in_ch, out_ch

		self.affine = nn.Linear(wdim, in_ch)
		# bias = 1 because w are weights so starts as identity
		self.affine.bias.data.fill_(1.0)

		# collection of convs, just using its weights
		# k, k is a kernel
		# (in, k, k) is a filter (produces one output channel)
		# (out, in, k, k) is a conv layer
		self.convs = nn.Conv2d(in_ch, out_ch, k, bias=False)

	
	def forward(self, img, w):
		assert w.shape[1] == self.wdim
		B, C, H, W = img.shape

		# dials on each of the in_ch's conv filter
		# (B, 512) -> (B, in_ch)
		s = self.affine(w)

		# dialing conv filters, modulate weights
		# (B, out, in, k, k) elem wise *
		# so entire (k, k) slice is scaled by learned.
		wt = self.convs.weight[None] * s[:, None, :, None, None]

		if self.demod:
			# demodulate factor
			# rescale filter length to 1, for relative emphasis not exploding/vanishing scaling of wt.
			# L2 norm over each filter (in, k, k)
			# 1 / sqrt( sum( wt^2 ) )
			# (B, out, in, k, k) -> (B, out)
			d = torch.rsqrt((wt**2).sum([2, 3, 4]) + 1e-8)

			# apply demod
			wt = wt * d[:, :, None, None, None]

		# grouped conv. do it all together in 1 batch, then decompress later.
		# because Con2d accepts (B, C, H, W) sizes and we want the GPU speed up
		# stack B imgs channels. so [RBGRGBRGB...]
		# (1, B*in, H, W)
		h = img.reshape(1, B*C, H, W)

		# stack B groups of out_ch filters [filter,filter...]
		# (B, out, in, k, k) -> (B * out, in, k, k)
		wt = wt.reshape(B * self.out_ch, C, self.k, self.k)

		# group B of out slides of group B of in
		# (1, B*in, H, W) *** (B*out, in, k, k) -> (1, B * out, H, W)
		h = F.conv2d(h, wt, padding=self.k // 2, groups=B)

		# (1, B*out, H, W) -> (B, out, H, W)
		h = h.reshape(B, self.out_ch, H, W)
		return h


class Generator(nn.Module):
	def __init__(self, wdim=512):
		super().__init__()
		self.wdim = wdim

		# blank canvas
		self.constant = nn.Parameter(torch.randn(1, wdim, 4, 4))

		# each layer *2^layer.
		# generally go down for compute as resolution grows. adds high res features.
		ch = [wdim, 512, 512, 256, 128, 64, 64]

		# weights, features, artistic flaire learned over time from dataset
		self.conv0 = ModConv(wdim, wdim)
		self.blocks = nn.ModuleList([
			# 1. cleans upsampling/bluring/crude enlargment into real features
			# 2. extract actual features at that resolution
			nn.ModuleList([ModConv(cin, cout), ModConv(cout, cout)]) for cin, cout in zip(ch, ch[1:])
		])

		# kernel size 1 so just in > out channel by one weight, H, W are the same
		self.torgb0 = ModConv(wdim, 3, k=1, demod=False)
		self.torgbs = nn.ModuleList([
			ModConv(cin, 3, k=1, demod=False) for cin in ch[1:]
		])
		
		self.final_resolution = 4 * 2**len(self.blocks)


	
	def forward(self, w):
		# (B, 512) paint this "instruction"
		B, wdim = w.shape
		assert wdim == self.wdim

		# (1, 512, 4, 4) -> (B, 512, 4, 4)
		# blank canvas
		x = self.constant.repeat(B, 1, 1, 1)


		# img's pixeldim grows at 4 * 2^blocks
		x = F.leaky_relu(self.conv0(x, w), 0.2)

		# project into (B, 3, 4, 4)
		rgb = self.torgb0(x, w)

		for (conv_a, conv_b), torgb in zip(self.blocks, self.torgbs):
			# begins small resolution/blurry, upsampling makes it more and more detailed
			# extract features according to latent w via a modulated Conv Layer
			x = F.interpolate(x, scale_factor=2)
			x = F.leaky_relu(conv_a(x, w), 0.2)
			x = F.leaky_relu(conv_b(x, w), 0.2)

			# generated features, at this resolution, in hyperspace
			# project hyperspace features into rgb
			rgb = F.interpolate(rgb, scale_factor=2) + torgb(x, w)

		return rgb

Generator()(torch.randn(2, 512)).shape # (2, 3, 128, 128)

torch.Size([2, 3, 256, 256])

In [16]:
class Discriminator(nn.Module):
	def __init__(self):
		super().__init__()

		ch = [32, 64, 128, 256, 512, 512, 512]

		self.conv0 = nn.Conv2d(3, ch[0], 1)
		self.blocks = nn.ModuleList([
			nn.ModuleList([nn.Conv2d(cin, cout, 3, padding=1), nn.Conv2d(cout, cout, 3, padding=1)])
			for cin, cout in zip(ch, ch[1:])
		])

		self.lrelu = nn.LeakyReLU(0.2)

		# scale the prev x up through the channels.
		# so we can add a res connection on each block. aka if a conv layer is useless then negate it
		self.skips = nn.ModuleList([
			nn.Conv2d(cin, cout, 1, bias=False) for cin, cout in zip(ch, ch[1:])
		])
		
		self.head = nn.Sequential(
			nn.Conv2d(ch[-1] + 1, ch[-1], 3, padding=1),
			nn.LeakyReLU(0.2),
			nn.Flatten(),
			nn.Linear(16 * ch[-1], ch[-1]),
			nn.LeakyReLU(0.2),
			nn.Linear(ch[-1], 1)
		)
	def mbstd(self, x, group=4):
		B, C, H, W = x.shape
		assert B % group == 0
		g = min(group, B)
		y = x.reshape(g, -1, C, H, W)            # split batch into groups
		y = y.std(dim=0).mean(dim=[1, 2, 3])     # avg std per group
		y = y.reshape(-1, 1, 1, 1).repeat(g, 1, H, W)
		return torch.cat([x, y], dim=1)          # +1 channel

	
	def forward(self, x):
		B, C, H, W = x.shape

		assert H == W
		assert 4 * 2**len(self.blocks) == W, "num of res blocks incorrect"

		# This is what we'll eventually mold to the (B, 512, 4, 4) shape which contains logit info
		# proj upwards from 3 to ch[0]
		# (B, 3, 256, 256) -> (B, 32, 256, 256)
		x = self.lrelu(self.conv0(x))

		for (conv_a, conv_b), skip_conv in zip(self.blocks, self.skips):
			# (B, 32, 256, 256) -> (B, 64, 256, 256)
			out = self.lrelu(conv_a(x))
			# (B, 64, 256, 256)
			out = self.lrelu(conv_b(out))
			# (B, 64, 256, 256) -> (B, 64, 128, 128)
			out = F.avg_pool2d(out, 2)

			# (B, 32, 256, 256) --pool--> (B, 32, 128, 128) --skipconv--> (B, 64, 128, 128)
			skip = skip_conv(F.avg_pool2d(x, 2)) 

			# just down sample and add skip res connection
			x = (skip + out) / math.sqrt(2)
		
		x = self.mbstd(x)
		
		# (B, 512, 4, 4) -> (B, 512, 4, 4) -> (B, 512 * 4 * 4) -> (B, 512) -> (B, 1)
		logit = self.head(x)

		return logit

(Discriminator()(torch.randn(4, 3, 256, 256))).shape # 4, 1

torch.Size([4, 1])

In [17]:
def augment(x, p):
    B = x.shape[0]
    def mask():  # which images get this op
        return (torch.rand(B, 1, 1, 1, device=x.device) < p).float()

    # x-flip
    x = torch.where(mask().bool(), x.flip(3), x)
    # translation (up to 1/8 of image), via roll
    if p > 0:
        shift = (torch.randint(-x.shape[3]//8, x.shape[3]//8 + 1, (B,)) * mask().view(B).long())
        x = torch.stack([torch.roll(xi, int(s), dims=2) for xi, s in zip(x, shift)])
    # brightness
    x = x + mask() * torch.randn(B, 1, 1, 1, device=x.device) * 0.2
    # contrast
    m = mask(); x = x * (1 - m + m * torch.exp2(torch.randn(B, 1, 1, 1, device=x.device) * 0.5))
    # saturation
    mean = x.mean(1, keepdim=True)
    m = mask(); x = mean + (x - mean) * (1 - m + m * torch.exp2(torch.randn(B, 1, 1, 1, device=x.device)))
    # cutout
    m = mask()
    H, W = x.shape[2:]
    cy, cx = torch.randint(0, H, (B,)), torch.randint(0, W, (B,))
    yy, xx = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    hole = ((yy[None] - cy[:, None, None]).abs() < H//4) & ((xx[None] - cx[:, None, None]).abs() < W//4)
    x = x * (1 - m * hole[:, None].float().to(x.device))
    return x

In [20]:
loader = [
	torch.randn(4, 3, 256, 256),
]

In [ ]:
bce = nn.BCEWithLogitsLoss()

gen = Generator()
discr = Discriminator()

opt_g = torch.optim.Adam(gen.parameters())
opt_d = torch.optim.Adam(discr.parameters())

p = 0.2

mapper = Mapping()

for real in loader:
	B, _, _, _ = real.shape

	z = torch.randn(B, 512)
	w = mapper(z)

	fake = gen(w).detach()

	a_real = augment(real, p) # avoid overfitting
	a_fake = augment(fake, p) # usually adaptive augmenting ADA so p changes

	# D step
	d_real = discr(a_real)
	d_fake = discr(a_fake)

	# all the real images passed through discr.... all the fake images passed through discr
	# 1 = real, 0 = fake (for discr)
	loss_discr = bce(d_real, torch.ones_like(d_real)) + bce(d_fake, torch.zeros_like(d_fake))

	opt_d.zero_grad()
	loss_discr.backward()
	opt_d.step()



	# G step
	fake = gen(w)
	a_fake = augment(fake, p)
	d_fake = discr(a_fake) # 1 = fake, 0 = real. minimax. for gen
	loss_gen = bce(d_fake, torch.ones_like(d_fake))

	opt_g.zero_grad()
	loss_gen.backward() # only backprop on gen's params
	opt_g.step()

	print(loss_discr.item(), loss_gen.item())
	
print("done")




1.386400818824768 0.6962659358978271
done
